# MotoGP Constructor Championships - Data Preparation

This notebook cleans and standardizes the constructor championship dataset following CRISP-DM methodology.

**Dataset Focus**: `constructure_world_championship.csv`  
**CRISP-DM Phase**: 3 - Data Preparation  
**Input**: Raw constructor championship data  
**Output**: Cleaned and prepared constructor championship data

## 0. Setup and Data Loading

In [ ]:
# Import necessary libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configure plot styles
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load raw data
raw_data_path = Path("../data/raw/constructure_world_championship.csv")
df_raw = pd.read_csv(raw_data_path)

print(f"Raw data loaded: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()

## 1. Data Quality Assessment

Based on exploration findings, assess data quality issues specific to constructor championships.

In [ ]:
# Assess data quality
print("=== DATA QUALITY ASSESSMENT ===")
print(f"Total championship records: {len(df_raw)}")
print(f"Season range: {df_raw['Season'].min()} - {df_raw['Season'].max()}")
print(f"Unique constructors: {df_raw['Constructor'].nunique()}")
print(f"Unique classes: {df_raw['Class'].nunique()}")

print(f"\nMissing values:")
print(df_raw.isnull().sum())

print(f"\nData types:")
print(df_raw.dtypes)

print(f"\nBasic statistics:")
print(df_raw.describe(include='all'))

## 2. Column Standardization

Standardize column names for consistency across the project.

In [ ]:
# Create working copy and standardize columns
df = df_raw.copy()

# Standardize column names
column_mapping = {
    'Season': 'season',
    'Constructor': 'constructor',
    'Class': 'class'
}

df = df.rename(columns=column_mapping)

print("Standardized columns:")
print(list(df.columns))
df.head()

## 3. Data Type Corrections

Ensure all columns have appropriate data types.

In [ ]:
# Ensure proper data types
df['season'] = df['season'].astype(int)

# Clean text fields
df['constructor'] = df['constructor'].astype(str).str.strip()
df['class'] = df['class'].astype(str).str.strip()

print("Updated data types:")
print(df.dtypes)

print(f"\nSeason range: {df['season'].min()} - {df['season'].max()}")
print(f"Total seasons: {df['season'].nunique()}")

## 4. Constructor Name Standardization

Clean and standardize constructor names for consistency.

In [ ]:
# Standardize constructor names
print("=== CONSTRUCTOR NAME STANDARDIZATION ===")
print(f"Original unique constructors: {df['constructor'].nunique()}")

# Clean constructor names
df['constructor_clean'] = df['constructor'].str.strip()

# Show constructor distribution
constructor_counts = df['constructor_clean'].value_counts()
print(f"\nConstructor championship distribution:")
print(constructor_counts)

# Manual corrections for common inconsistencies
constructor_corrections = {
    # Add specific corrections if found during exploration
    # Example: 'Old Name': 'Standardized Name'
}

if constructor_corrections:
    df['constructor_clean'] = df['constructor_clean'].replace(constructor_corrections)
    print(f"Applied {len(constructor_corrections)} constructor name corrections")

print(f"\nFinal unique constructors: {df['constructor_clean'].nunique()}")

## 5. Class Standardization and Categorization

Standardize racing class names and create broader categories.

In [ ]:
# Standardize racing classes
print("=== CLASS STANDARDIZATION ===")

df['class_clean'] = df['class'].str.strip()

print(f"Class distribution:")
class_counts = df['class_clean'].value_counts()
print(class_counts)

# Create broader class categories for analysis
def categorize_class(class_name):
    class_name = str(class_name).upper()
    if 'MOTOGP' in class_name or '500' in class_name:
        return 'Premier Class'
    elif 'MOTO2' in class_name or '250' in class_name:
        return 'Intermediate Class'
    elif 'MOTO3' in class_name or '125' in class_name:
        return 'Lightweight Class'
    elif 'MOTOE' in class_name:
        return 'Electric Class'
    else:
        return 'Other Class'

df['class_category'] = df['class_clean'].apply(categorize_class)

print(f"\nClass categories:")
print(df['class_category'].value_counts())

# Show constructor success by class category
print(f"\nConstructor championships by category:")
category_breakdown = pd.crosstab(df['class_category'], df['constructor_clean'], margins=True)
print(category_breakdown)

## 6. Temporal Analysis and Grouping

Add temporal groupings for era-based analysis.

In [ ]:
# Add temporal groupings
print("=== TEMPORAL ANALYSIS ===")

# Create decade grouping
df['decade'] = (df['season'] // 10) * 10

# Create era grouping
def categorize_era(season):
    if season < 1960:
        return 'Early Era (1949-1959)'
    elif season < 1980:
        return 'Classic Era (1960-1979)'
    elif season < 2000:
        return 'Modern Era (1980-1999)'
    elif season < 2020:
        return 'Contemporary Era (2000-2019)'
    else:
        return 'Current Era (2020+)'

df['era'] = df['season'].apply(categorize_era)

print(f"Championships by decade:")
decade_counts = df['decade'].value_counts().sort_index()
print(decade_counts)

print(f"\nChampionships by era:")
era_counts = df['era'].value_counts()
print(era_counts)

# Top constructors by era
print(f"\nTop constructor by era:")
for era in sorted(df['era'].unique()):
    era_data = df[df['era'] == era]
    if len(era_data) > 0:
        top_constructor = era_data['constructor_clean'].value_counts().head(1)
        print(f"{era}: {top_constructor.index[0]} ({top_constructor.iloc[0]} titles)")

## 7. Constructor Performance Metrics

Calculate comprehensive performance metrics for constructors.

In [ ]:
# Calculate constructor performance metrics
print("=== CONSTRUCTOR PERFORMANCE METRICS ===")

# Aggregate metrics by constructor
constructor_metrics = df.groupby('constructor_clean').agg({
    'season': ['count', 'min', 'max', 'nunique'],
    'class_clean': 'nunique',
    'class_category': 'nunique'
})

constructor_metrics.columns = [
    'total_championships', 'first_championship', 'last_championship', 
    'seasons_active', 'classes_won', 'categories_won'
]

# Calculate additional metrics
constructor_metrics['championship_span'] = constructor_metrics['last_championship'] - constructor_metrics['first_championship']
constructor_metrics['avg_championships_per_active_season'] = (
    constructor_metrics['total_championships'] / constructor_metrics['seasons_active']
)

# Add versatility indicator
constructor_metrics['is_versatile'] = constructor_metrics['categories_won'] > 1

# Sort by total championships
constructor_metrics = constructor_metrics.sort_values('total_championships', ascending=False)

print("Top 10 constructor performance metrics:")
print(constructor_metrics.head(10))

# Most versatile constructors
versatile_constructors = constructor_metrics[constructor_metrics['is_versatile']]
print(f"\nMost versatile constructors (multiple categories):")
print(versatile_constructors[['total_championships', 'categories_won', 'classes_won']].head())

## 8. Data Validation

Comprehensive validation of the prepared data.

In [ ]:
# Comprehensive data validation
print("=== DATA VALIDATION ===")

# Check season ranges
valid_seasons = (df['season'] >= 1949) & (df['season'] <= 2025)
invalid_seasons = (~valid_seasons).sum()
if invalid_seasons > 0:
    print(f"Warning: {invalid_seasons} championships with invalid seasons")
else:
    print("✓ All seasons are within expected range")

# Check for missing data in required fields
required_fields = ['season', 'constructor_clean', 'class_clean']
for field in required_fields:
    missing = df[field].isnull().sum()
    if missing > 0:
        print(f"Warning: {missing} missing values in {field}")
    else:
        print(f"✓ No missing values in {field}")

# Check for duplicate championships (should be unique per season-class combination)
duplicates = df.duplicated(subset=['season', 'class_clean']).sum()
if duplicates > 0:
    print(f"Warning: {duplicates} duplicate season-class combinations")
    duplicate_examples = df[df.duplicated(subset=['season', 'class_clean'], keep=False)]
    print("Examples:")
    print(duplicate_examples[['season', 'class_clean', 'constructor_clean']].head())
else:
    print("✓ No duplicate season-class combinations")

print(f"\n=== VALIDATION SUMMARY ===")
print(f"Total championship records after preparation: {len(df)}")
print(f"Season range: {df['season'].min()} - {df['season'].max()}")
print(f"Unique constructors: {df['constructor_clean'].nunique()}")
print(f"Unique classes: {df['class_clean'].nunique()}")
print(f"Unique seasons: {df['season'].nunique()}")

## 9. Final Dataset Structure

Create the final prepared dataset with consistent structure.

In [ ]:
# Define final column order
final_columns = [
    # Core data
    'season', 'constructor', 'constructor_clean', 'class', 'class_clean', 'class_category',
    # Temporal groupings
    'decade', 'era'
]

# Create final dataset
df_final = df[final_columns].copy()

print("=== FINAL PREPARED DATASET ===")
print(f"Shape: {df_final.shape}")
print(f"Columns: {list(df_final.columns)}")

print(f"\nSample of prepared data:")
print(df_final.head(15))

print(f"\nData types:")
print(df_final.dtypes)

print(f"\nMissing values:")
print(df_final.isnull().sum())

## 10. Save Prepared Data

Save the cleaned and prepared constructor championship dataset.

In [ ]:
# Create prepared data directory if it doesn't exist
prepared_data_path = Path("../../data/interim/")
prepared_data_path.mkdir(parents=True, exist_ok=True)

# Save prepared dataset
output_file = prepared_data_path / "constructors_prepared.csv"
df_final.to_csv(output_file, index=False)

print(f"Prepared dataset saved to: {output_file}")
print(f"File size: {output_file.stat().st_size:,} bytes")

# Also save constructor metrics for reference
constructor_metrics_file = prepared_data_path / "constructor_performance_metrics.csv"
constructor_metrics.to_csv(constructor_metrics_file)
print(f"Constructor metrics saved to: {constructor_metrics_file}")

# Verification
df_verification = pd.read_csv(output_file)
print(f"\nVerification - loaded shape: {df_verification.shape}")
print("✓ Files saved and verified successfully")

## 11. Preparation Summary

Summary of data preparation steps and improvements made to the constructor championship dataset.

In [ ]:
print("=== PREPARATION SUMMARY ===")
print("\n✅ COMPLETED TASKS:")
print("1. Column name standardization")
print("2. Data type corrections and validation")
print("3. Constructor name cleaning and standardization")
print("4. Racing class standardization and categorization")
print("5. Temporal groupings (decades, eras)")
print("6. Constructor performance metrics calculation")
print("7. Data validation and consistency checks")
print("8. Prepared dataset and metrics saved")

print("\n📊 DATASET IMPROVEMENTS:")
print(f"- Standardized constructor and class names")
print(f"- Added class categories for broader analysis")
print(f"- Added temporal groupings for era-based analysis")
print(f"- Generated comprehensive performance metrics")
print(f"- Validated all data relationships")

print("\n🏆 CONSTRUCTOR DOMINANCE:")
top_constructors = df_final['constructor_clean'].value_counts().head(5)
for constructor, count in top_constructors.items():
    percentage = (count / len(df_final)) * 100
    print(f"- {constructor}: {count} championships ({percentage:.1f}%)")

print("\n🏁 CLASS DISTRIBUTION:")
class_dist = df_final['class_category'].value_counts()
for category, count in class_dist.items():
    percentage = (count / len(df_final)) * 100
    print(f"- {category}: {count} championships ({percentage:.1f}%)")

print("\n📅 TEMPORAL COVERAGE:")
era_dist = df_final['era'].value_counts()
for era, count in era_dist.items():
    percentage = (count / len(df_final)) * 100
    print(f"- {era}: {count} championships ({percentage:.1f}%)")

print("\n🔍 KEY INSIGHTS:")
most_successful_constructor = df_final['constructor_clean'].value_counts().index[0]
most_successful_count = df_final['constructor_clean'].value_counts().iloc[0]
print(f"- Most successful constructor: {most_successful_constructor} with {most_successful_count} championships")

most_competitive_class = df_final['class_category'].value_counts().index[0]
most_competitive_count = df_final['class_category'].value_counts().iloc[0]
print(f"- Most competitive class category: {most_competitive_class} with {most_competitive_count} championships")

print("\n➡️  READY FOR MODELING PHASE")
print("The prepared constructor championship dataset is now ready for advanced modeling and analysis.")